# Compute fitness effects for subset trees

Use expected rates from the global neutral model to compute fitness effects for each subset (host, geographic, split-half).

In [1]:
import pandas as pd
import numpy as np

## Read data

In [2]:
# Read subset counts
counts_df = pd.read_csv('../results/subset_counts.csv', keep_default_na=False, low_memory=False)
counts_df = counts_df.replace('', np.nan)
counts_df['codon_site'] = counts_df['codon_site'].astype(str)
print(f'Subset counts: {len(counts_df):,} rows')
print(f'Subsets: {sorted(counts_df["subset"].unique())}')

Subset counts: 4,357,290 rows
Subsets: ['asia', 'avian', 'europe', 'human', 'north_america', 'split_a', 'split_b', 'swine']


In [3]:
# Read expected rates from the global neutral model
predicted_rates_df = pd.read_csv('../results/expected_rates.csv', keep_default_na=False)
del predicted_rates_df['tau_squared']
predicted_rates_df.head()

,mut_type,segment,motif,predicted_rate
0,AC,HA,AAA,0.200380
1,AC,HA,AAC,0.315456
2,AC,HA,AAG,0.246664
3,AC,HA,AAT,0.308056
4,AC,HA,CAA,0.178563


## Compute expected counts and filter

In [4]:
counts_cutoff = 10

actual_expected_df = (
    counts_df
    .merge(predicted_rates_df, on=['mut_type', 'segment', 'motif'], how='left')
    .assign(expected_count=lambda x: x['predicted_rate'] * x['evo_opp'])
    .query('actual_count >= @counts_cutoff | expected_count >= @counts_cutoff')
)
print(f'Rows after filtering (>= {counts_cutoff} actual or expected): {len(actual_expected_df):,}')
actual_expected_df.head()

Rows after filtering (>= 10 actual or expected): 265,953


,site,nt_mut,wt_nt,mut_nt,gene,codon_position,codon_site,wt_codon,mut_codon,wt_aa,...,subtype,segment,segment_subtype,segment_length,subset,subset_type,evo_opp,rate,predicted_rate,expected_count
52,54,A54C,A,C,NP,3,18,GAA,GAC,E,...,all,NP,NP_all,1494,avian,host,21.074297,0.6643163411148166,0.338733,7.138558
110,59,A59C,A,C,NP,2,20,CAG,CCG,Q,...,all,NP,NP_all,1494,human,host,57.072959,0.3854715188760013,0.236027,13.470734
233,747,A747C,A,C,NA,3,249,AAA,AAC,K,...,N2,NA,NA_N2,1407,human,host,36.302061,0.08263993578322924,0.275654,10.006795
241,985,A985C,A,C,NA,1,329,ACC,CCC,T,...,N2,NA,NA_N2,1407,split_b,split_half,0.874200,12.582926829268292,0.352531,0.308183
453,986,A986C,A,C,NA,2,329,AAC,ACC,N,...,N2,NA,NA_N2,1407,split_b,split_half,7.432125,9.284020273501005,0.352531,2.620058


## Compute amino-acid-level fitness effects

In [5]:
pseudo_count = 0.5

groupby_cols = [
    'subset', 'subset_type', 'subtype', 'segment', 'gene',
    'codon_site', 'wt_aa', 'mut_aa', 'aa_mut', 'mut_class'
]

aa_fitness_df = (
    actual_expected_df
    .groupby(groupby_cols, as_index=False)
    .agg({'actual_count': 'sum', 'expected_count': 'sum'})
    .assign(
        delta_fitness=lambda x:
            np.log((x['actual_count'] + pseudo_count) / (x['expected_count'] + pseudo_count))
    )
)

aa_fitness_df.to_csv('../results/subset_aa_fitness_effects.csv', index=False)
print(f'AA-level fitness effects: {len(aa_fitness_df):,} rows')
aa_fitness_df.head()

AA-level fitness effects: 185,492 rows


,subset,subset_type,subtype,segment,gene,codon_site,wt_aa,mut_aa,aa_mut,mut_class,actual_count,expected_count,delta_fitness
0,asia,geographic,H1,HA,HA,1,M,I,M1I,nonsynonymous,0,30.581079,-4.129746
1,asia,geographic,H1,HA,HA,1,M,T,M1T,nonsynonymous,0,13.846674,-3.356665
2,asia,geographic,H1,HA,HA,10,Y,C,Y10C,nonsynonymous,26,23.617486,0.094208
3,asia,geographic,H1,HA,HA,10,Y,H,Y10H,nonsynonymous,21,26.990156,-0.245775
4,asia,geographic,H1,HA,HA,10,Y,Y,Y10Y,synonymous,81,29.269693,1.007112


In [6]:
# Summary by subset and mutation class
aa_fitness_df.groupby(['subset', 'mut_class']).size().unstack(fill_value=0)

mut_class,nonsense,nonsynonymous,synonymous
subset,,,
asia,845,16615,5963
avian,738,13571,4793
europe,811,15232,5440
human,1636,23356,5400
north_america,1076,18213,5761
split_a,1369,21079,6015
split_b,1373,21148,6021
swine,210,6041,2786
